# VoiceSecure 훈련 노트북

**실행 순서대로 셀을 실행하세요.**

사전 준비:
- Google Drive에 KSS 데이터셋 폴더를 업로드해두세요.
- 런타임 유형을 **GPU (T4 이상)** 으로 설정하세요.

## 1. GPU 확인

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU를 사용할 수 없습니다. 런타임 > 런타임 유형 변경 > GPU 로 설정하세요.')

## 2. Google Drive 마운트

아래 셀 실행 후 Drive 연결 허용을 눌러주세요.  
KSS 데이터셋 경로를 `KSS_DIR` 변수에 본인 Drive 경로로 수정하세요.

예) Drive에 `내 드라이브/kss` 폴더로 올렸다면 → `/content/drive/MyDrive/kss`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ↓ 본인 Drive의 KSS 데이터셋 경로로 수정하세요
KSS_DIR = '/content/drive/MyDrive/kss'

## 3. CosyVoice 레포 클론 및 의존성 설치

In [ ]:
import os

if not os.path.exists('CosyVoice'):
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git
else:
    print('CosyVoice 이미 존재, 스킵')

In [ ]:
!pip install -r CosyVoice/requirements.txt -q

## 4. CosyVoice3 모델 다운로드

In [ ]:
if not os.path.exists('CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B-2512'):
    from huggingface_hub import snapshot_download
    snapshot_download(
        'FunAudioLLM/Fun-CosyVoice3-0.5B-2512',
        local_dir='CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B-2512'
    )
else:
    print('모델 이미 존재, 스킵')

## 5. VoiceSecure SDK 클론 및 설치

In [ ]:
if not os.path.exists('voicesecure-sdk'):
    !git clone https://github.com/VoiceSecureHoseo/voicesecure-sdk.git
else:
    print('voicesecure-sdk 이미 존재, 스킵')

%cd voicesecure-sdk
!pip install -e . -q

## 6. 추가 의존성 설치

- `tensorboard`: 훈련 로그 시각화
- `soundfile`: 오디오 파일 입출력
- `librosa`: 음성 특징 추출 (MFCC, F0 등)
- `transformers`: WavLM-SV (`microsoft/wavlm-base-plus-sv`) 포함  
  → WavLM 모델 weights는 훈련 시작 시 HuggingFace에서 자동 다운로드됩니다.

In [ ]:
!pip install tensorboard soundfile librosa transformers -q

## 7. 훈련 실행

훈련 옵션:
- `--epochs`: 훈련 에폭 수 (기본 3)
- `--checkpoint_interval`: 체크포인트 저장 주기 에피소드 수 (기본 1000)
- `--resume`: 이어서 훈련할 체크포인트 경로 (선택)

체크포인트와 로그는 `voicesecure-sdk/checkpoints/` 에 저장됩니다.

In [ ]:
import sys
sys.path.insert(0, '/content/CosyVoice')
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')

!python train.py \
    --data_dir {KSS_DIR} \
    --model_dir ../CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B-2512 \
    --cosyvoice_root ../CosyVoice \
    --epochs 3

## 8. TensorBoard 로그 확인 (선택)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir checkpoints/logs

## 9. 체크포인트 Drive에 백업 (선택)

Colab은 런타임 종료 시 파일이 삭제됩니다.  
중요한 체크포인트는 Drive에 백업하세요.

In [ ]:
import shutil

# ↓ Drive 백업 경로 수정
BACKUP_DIR = '/content/drive/MyDrive/voicesecure_checkpoints'

shutil.copytree('checkpoints', BACKUP_DIR, dirs_exist_ok=True)
print(f'체크포인트 백업 완료: {BACKUP_DIR}')